In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print("✅ Working directory set to project root:", PROJECT_ROOT)


In [ ]:
import pandas as pd

In [ ]:
DATAFOLDER = PROJECT_ROOT / "data"
RAW_FOLDER = DATAFOLDER / "raw"
EXTRACTED_FOLDER = DATAFOLDER / "extracted"
RAW_FOLDER.mkdir(parents=True, exist_ok=True)
EXTRACTED_FOLDER.mkdir(parents=True, exist_ok=True)
print("✅ Data folders set up at:", DATAFOLDER)

In [ ]:
FILES = {
    "2025_Q1": RAW_FOLDER / "personal_wellbeing_loneliness_q1_2025.xlsx",
    "2025_Q2": RAW_FOLDER / "personal_wellbeing_loneliness_q2_2025.xlsx",
    "2025_Q3": RAW_FOLDER / "personal_wellbeing_loneliness_q3_2025.xlsx",
    "2025_Q4": RAW_FOLDER / "personal_wellbeing_loneliness_q4_2025.xlsx",
}

In [ ]:
METRICS = {
    "Table_1": "life_satisfaction",
    "Table_2": "worthwhile",
    "Table_3": "happiness",
    "Table_4": "anxiety",
}

In [ ]:
SKIPROWS = 9  # keep consistent with what you used

In [ ]:
def extract_region_mean(path: Path, sheet: str, metric_name: str, quarter: int) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet, skiprows=SKIPROWS)

    # Keep only Region, Country, All persons
    df = df[
        df["Breakdown category"].isin(["Region", "Country", "All persons"])
    ][["Breakdown category", "Breakdown", "Estimate (mean score out of 10)"]].copy()

    # Rename national row
    df.loc[df["Breakdown category"] ==
           "All persons", "Breakdown"] = "Great Britain"

    # Rename columns
    df.columns = ["geography_level", "geography", metric_name]

    # Clean values
    df["geography"] = df["geography"].astype(str).str.strip()
    df[metric_name] = pd.to_numeric(df[metric_name], errors="coerce")

    # Map geography_level to clean labels
    df["geography_level"] = df["geography_level"].replace({
        "Region": "region",
        "Country": "country",
        "All persons": "national"
    })

    df["quarter"] = quarter

    # Sanity check (should now be 13 rows)
    if df["geography"].nunique() != 13:
        print(
            f"⚠️ {quarter} {sheet}: expected 13 geographies, got {df['geography'].nunique()}")

    return df[["quarter", "geography_level", "geography", metric_name]]

In [ ]:
def build_quarter_means(path: Path, quarter: int) -> pd.DataFrame:
    """
    Builds one quarter of mean data across region/country/national:
    returns: quarter, geography_level, geography, life_satisfaction, worthwhile, happiness, anxiety
    """
    quarter_df = None

    for sheet, metric_name in METRICS.items():
        metric_df = extract_region_mean(path, sheet, metric_name, quarter)
        # metric_df columns: quarter, geography_level, geography, <metric_name>

        if quarter_df is None:
            quarter_df = metric_df
        else:
            quarter_df = quarter_df.merge(
                metric_df,
                on=["quarter", "geography_level", "geography"],
                how="inner"
            )

    # Optional: consistent sorting
    quarter_df = quarter_df.sort_values(
        ["geography_level", "geography"]).reset_index(drop=True)

    return quarter_df

In [ ]:
all_quarters = [build_quarter_means(path, q) for q, path in FILES.items()]
final_means = pd.concat(all_quarters, ignore_index=True)

final_means["wellbeing_index"] = (
    final_means["life_satisfaction"]
    + final_means["worthwhile"]
    + final_means["happiness"]
    - final_means["anxiety"]
) / 4

final_means["wellbeing_index"] = final_means["wellbeing_index"].round(2)

In [ ]:
print(final_means.shape)          # should be (36, 6)
print(final_means.head())

In [ ]:
# Save one clean file
OUT_PATH = EXTRACTED_FOLDER / "means_quarterly.csv"
final_means.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

In [ ]:
final_means.shape        

In [ ]:
          # should be (36, 6)
final_means["quarter"].nunique()   

In [ ]:
# should be 4
final_means.groupby("quarter")["geography"].nunique()  # should be 9 for each


In [ ]:
final_means[["life_satisfaction","worthwhile","happiness","anxiety"]].describe()

In [ ]:
# should be 4
final_means[["life_satisfaction","worthwhile","happiness","anxiety","wellbeing_index"]].corr()